In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a program that performs <code>R</code> rounds of parallel hashing on an array of 32-bit integers using the provided hash function.
    The hash should be applied <code>R</code> times iteratively (the output of one round becomes the input to the next).
</p>

<h2>Implementation Requirements</h2>
<ul>
    <li>External libraries are not permitted</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The final result must be stored in array <code>output</code></li>
</ul>

<h2>Example 1:</h2>
<pre>Input:  numbers = [123, 456, 789], R = 2
Output: hashes = [1636807824, 1273011621, 2193987222]</pre>

<h2>Example 2:</h2>
<pre>Input:  numbers = [0, 1, 2147483647], R = 3
Output: hashes = [96754810, 3571711400, 2006156166]</pre>

<h2>Constraints</h2>
<ul>
    <li>1 ≤ <code>N</code> ≤ 10,000,000</li>
    <li>1 ≤ <code>R</code> ≤ 100</li>
    <li>0 ≤ <code>input[i]</code> ≤ 2147483647</li>

  <li>Performance is measured with <code>N</code> = 5,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__device__ unsigned int fnv1a_hash(unsigned int input) {
    const unsigned int FNV_PRIME = 16777619;
    const unsigned int OFFSET_BASIS = 2166136261;

    unsigned int hash = OFFSET_BASIS;

    for (int byte_pos = 0; byte_pos < 4; byte_pos++) {
        unsigned char byte = (input >> (byte_pos * 8)) & 0xFFu;
        hash = (hash ^ byte) * FNV_PRIME;
    }

    return hash;
}

__global__ void fnv1a_hash_kernel(const int* input, unsigned int* output, int N, int R) {}

// input, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const int* input, unsigned int* output, int N, int R) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    fnv1a_hash_kernel<<<blocksPerGrid, threadsPerBlock>>>(input, output, N, R);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


def fnv1a_hash_u32_scalar(x: cute.Uint32) -> cute.Uint32:
    FNV_PRIME = 16777619
    OFFSET_BASIS = 2166136261
    hash_val = cute.Uint32(OFFSET_BASIS)
    prime = cute.Uint32(FNV_PRIME)
    mask = cute.Uint32(0xFF)
    for byte_pos in range(4):
        byte = (x >> (byte_pos * 8)) & mask
        hash_val = (hash_val ^ byte) * prime
    return cute.Uint32(hash_val)


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, N: cute.Int32, R: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


def fnv1a_hash(x: jax.Array) -> jax.Array:
    FNV_PRIME = jnp.uint32(16777619)
    OFFSET_BASIS = jnp.uint32(2166136261)
    hash_val = jnp.full_like(x, OFFSET_BASIS, dtype=jnp.uint32)

    MASK_FF = jnp.uint32(0xFF)
    for byte_pos in range(4):
        byte = (x >> jnp.uint32(byte_pos * 8)) & MASK_FF
        hash_val = hash_val ^ byte
        hash_val = hash_val * FNV_PRIME

    return hash_val


# input is a tensor on the GPU
def solve(input: jax.Array, N: int, R: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def fnv1a_hash(input: UInt32) -> UInt32:
    alias FNV_PRIME: UInt32 = 16777619
    alias OFFSET_BASIS: UInt32 = 2166136261

    var hash: UInt32 = OFFSET_BASIS

    for byte_pos in range(4):
        var byte_val: UInt32 = (input >> (byte_pos * 8)) & UInt32(0xFF)
        hash = (hash ^ byte_val) * FNV_PRIME

    return hash


def fnv1a_hash_kernel(
    input: UnsafePointer[Int32, MutExternalOrigin],
    output: UnsafePointer[UInt32, MutExternalOrigin],
    N: Int32,
    R: Int32,
):
    pass


# input, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    input: UnsafePointer[Int32, MutExternalOrigin],
    output: UnsafePointer[UInt32, MutExternalOrigin],
    N: Int32,
    R: Int32,
) raises:
    var threadsPerBlock: Int32 = 256
    var ctx = DeviceContext()

    var blocksPerGrid = ceildiv(N, threadsPerBlock)

    var _kernel = ctx.compile_function[fnv1a_hash_kernel, fnv1a_hash_kernel]()
    ctx.enqueue_function(
        _kernel, input, output, N, R, grid_dim=blocksPerGrid, block_dim=threadsPerBlock
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


def fnv1a_hash(x: torch.Tensor) -> torch.Tensor:
    FNV_PRIME = 16777619
    OFFSET_BASIS = 2166136261
    x_int = x.to(torch.int64)
    hash_val = torch.full_like(x_int, OFFSET_BASIS, dtype=torch.int64)

    for byte_pos in range(4):
        byte = (x_int >> (byte_pos * 8)) & 0xFF
        hash_val = (hash_val ^ byte) * FNV_PRIME
        hash_val = hash_val & 0xFFFFFFFF

    return hash_val.to(torch.int32)


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int, R: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


@triton.jit
def fnv1a_hash(x):
    FNV_PRIME = 16777619
    OFFSET_BASIS = 2166136261

    hash_val = tl.full(x.shape, OFFSET_BASIS, tl.uint32)

    for byte_pos in range(4):
        byte = (x >> (byte_pos * 8)) & 0xFF
        hash_val = (hash_val ^ byte) * FNV_PRIME

    return hash_val


@triton.jit
def fnv1a_hash_kernel(input, output, n_elements, n_rounds, BLOCK_SIZE: tl.constexpr):
    pass


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int, R: int):
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(N, BLOCK_SIZE),)
    fnv1a_hash_kernel[grid](input, output, N, R, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/easy/24_rainbow_table/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
